In [5]:
using NNlib
using PyCall
using IterativeSolvers
using LinearMaps
using BenchmarkTools
np=pyimport("numpy")

const base_length = [16,16,16]

3-element Vector{Int64}:
 16
 16
 16

In [27]:
function apply_stencil(u; diag = 1)
    i, j, k = size(u)
    result = zeros(Float64, size(u))
    result[2:end-1, 2:end-1, 2:end-1] .= (
        .+ 6 .* u[2:end-1, 2:end-1, 2:end-1]
        .- u[3:end, 2:end-1, 2:end-1] .- u[1:end-2, 2:end-1, 2:end-1]
        .- u[2:end-1, 3:end, 2:end-1] .- u[2:end-1, 1:end-2, 2:end-1]
        .- u[2:end-1, 2:end-1, 3:end] .- u[2:end-1, 2:end-1, 1:end-2]
        
    ) / diag
    return result
end

function subtract_apply_stencil(result, u, rhs; diag)
    result[2:end-1, 2:end-1, 2:end-1] = rhs[2:end-1, 2:end-1, 2:end-1] .- (
        + 6*u[2:end-1, 2:end-1, 2:end-1]
        - u[3:end, 2:end-1, 2:end-1] - u[1:end-2, 2:end-1, 2:end-1]
        - u[2:end-1, 3:end, 2:end-1] - u[2:end-1, 1:end-2, 2:end-1]
        - u[2:end-1, 2:end-1, 3:end] - u[2:end-1, 2:end-1, 1:end-2]
        
    ) ./ diag
end


function compute_res(u, rhs, diag)
    #print("The factor diag in compute res is: ", diag)
    temp = apply_stencil(u; diag = diag) .- rhs
    return sqrt(sum(temp .* temp)) / sqrt(sum(rhs .* rhs))
end


function rb_gauss_seidel_3d(u::Array{Float64, 3}, f::Array{Float64, 3}; diag::Real =1, iterations::Int=3)
    #print("The factor diag in rb_gauss_seidel_3d is: ", diag)
    n, m, p = size(u)
    for _ = 1:iterations
        # Red update: (i + j + k) % 2 == 0
           Threads.@threads for k = 2:p-1
            for j = 2: m - 1 
                 @simd for i = 2:n - 1
                    if (i + j + k) % 2 == 1
                        u[i, j, k] = (1/6) * (
                            u[i+1, j, k] + u[i-1, j, k] +
                            u[i, j+1, k] + u[i, j-1, k] +
                            u[i, j, k+1] + u[i, j, k-1] +
                            diag * f[i, j, k]
                        )

                    end
                end
            end
        end
           # Black update: (i + j + k) % 2 == 1
         Threads.@threads for k=2:p-1
             for j=2:m-1
               @simd for i=2:n - 1
                    if (i + j + k) % 2 == 0
                        u[i, j, k] =  (1/6) * (
                            u[i+1, j, k] + u[i-1, j, k] +
                            u[i, j+1, k] + u[i, j-1, k] +
                            u[i, j, k+1] + u[i, j, k-1] +
                            diag * f[i, j, k]
                        )
                    end
                end
           end
        end
    end
        
end


function restrict(res)
    return res[1:2:end, 1:2:end, 1:2:end]
end

function prolong(coarse)
    i, j, k = size(coarse)
    temp = reshape(coarse, (size(coarse)..., 1, 1))
    temp2 = upsample_trilinear(temp; size=(2*i-1, 2*j-1, 2*k-1))
    
    return reshape(temp2, (2*i-1, 2*j-1, 2*k-1))
end

prolong (generic function with 1 method)

In [28]:
function set_boundary_conditions(u)
    i, j, k = size(u)
    for x =1:i
        for y = 1:j
            u[x, y, end]=1
            u[x, y, 1] = 0
        end
    end

    for x =1:i
        for z =1:k
            u[x, end, z] = (z-1) / (k-1)
            u[x, 1, z] = (z-1) / (k-1)
        end 
    end

    for y = 1:j
        for z = 1:k
            u[end, y, z] = (z-1) / (k-1)
            u[1, y, z] = (z-1) / (k-1)
        end
    end
end

function matvec_shape(x_flat, shape, x_grid; factor=2)
    #x_grid = zeros(Float64,shape[1]+2, shape[2]+2, shape[3]+2)
    x_grid[2:end-1, 2:end-1, 2:end-1] = reshape(x_flat, shape)
    Ax_grid = apply_stencil(x_grid) ./ factor
    return vec(Ax_grid[2:end-1, 2:end-1, 2:end-1])
end

matvec_shape (generic function with 1 method)

In [46]:
function multigrid_V_cycle(base_length; num_iters=[1 , 1], num_smooth = 2, num_levels = nothing, level_num = nothing, rhs = nothing)

    if (num_levels===nothing || level_num===nothing)
        num_levels=level_num=size(num_iters)[1]
    end

    if(size(num_iters)[1] == 1)
        num_iters = [num_iters[1] for i = 1:num_levels]
    end
    
    length_by_level = [2^n .* base_length .+ 1 for n=1:level_num + 1]

    #Allocating working array
    guess = zeros(Float64, length_by_level[level_num]...)
    u = zeros(Float64, length_by_level[level_num]...)

    #if in uppermost level, set up the problem
    if rhs === nothing       
        #setting boundary conditions       
        set_boundary_conditions(u)
        #print(u)
        #Allocating right hand side
        rhs = -apply_stencil(u)
    end



    #getting matvec function and linear operator
    if level_num == 1
         x_grid = zeros(Float64, length_by_level[1]...)
         linear_function = x_flat -> matvec_shape(x_flat, Tuple(length_by_level[1] .- 2), x_grid; factor = 2^(num_levels-level_num))
         n = (2*base_length[1]-1)*(2*base_length[2]-1)*(2*base_length[3]-1)
         A = FunctionMap{Float64,false}(linear_function, n, n);
         unravel_c = reshape(rhs[2:end-1, 2:end-1, 2:end-1], :)
         #solving #coarse_grid
         sol = cg(A, unravel_c)
         #print (sol)
         sol_shaped = reshape(sol, Tuple(2 .* base_length .- 1))
         rhs[2:end-1, 2:end-1, 2:end-1] = sol_shaped
        
         return rhs
    end




    #Now doing the multigrid operation
    defect_domain=zeros(size(rhs))
    residuals = zeros(num_iters[1])
    for i = 1:num_iters[1]
        if level_num==num_levels
            residuum = compute_res(guess, rhs, 2^(num_levels-level_num))
            println("After ", i, " iterations the residual is ", residuum)
            residuals[i]=residuum
        end

        #print("The current level is: ", level_num)
        rb_gauss_seidel_3d(guess, rhs; diag = 2^(num_levels-level_num), iterations = num_smooth)
        #print("The guess after the rb_guass_seidel is: ")
        #print(guess)
        #print("The right hand side for the ")
        #print("The guess after a ", 2, " Gauss-Seidel iteration is: ")
        #print(guess)
        #coarsening
        subtract_apply_stencil(defect_domain, guess, rhs; diag = 2^(num_levels-level_num))
        #print("The defect domain before coarsening is: ")
        #print(defect_domain)
        coarse = restrict(defect_domain)
        coarse = multigrid_V_cycle(base_length; num_iters=num_iters[2:end] ,num_smooth=num_smooth ,num_levels = num_levels, 
        level_num = level_num-1, rhs = coarse)

        #refining
       
        #refinemend = prolong(coarse)

        #print("The refined coarse grid solution: ")
        #print(refinemend)

        #Adding correction
        #print (refinemend.shape)
        guess[2:end-1, 2:end-1, 2:end-1] .+= prolong(coarse)[2:end-1, 2:end-1, 2:end-1]

        #print("After adding the coarse grid correction the guess is:")
        #print(guess)
        
        #doing the post smoothing
        #print("Current level: ", level_num)
        rb_gauss_seidel_3d(guess, rhs; diag= 2^(num_levels-level_num), iterations= num_smooth)

        #print("After the post smoothing step the guess is:")
        #print(guess)
    end
    
    guess .+= u


    return level_num==num_levels ? (guess, residuals) : guess
end

multigrid_V_cycle (generic function with 1 method)

In [48]:
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.05213625757457693
After 3 iterations the residual is 0.0016434398572336764
After 4 iterations the residual is 5.724398881086094e-5
After 5 iterations the residual is 2.002917113027301e-6
After 6 iterations the residual is 7.536033202713814e-8
After 7 iterations the residual is 2.7747487985960133e-9
After 8 iterations the residual is 1.1304617064125096e-10
After 9 iterations the residual is 4.546352910637512e-12
After 10 iterations the residual is 2.1007444694301508e-13
  1.096970 seconds (50.08 k allocations: 3.353 GiB, 22.48% gc time)


In [50]:
#base_length = [16,16,16]
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.05745543683141366
After 3 iterations the residual is 0.0019321056344463493
After 4 iterations the residual is 6.63013578239215e-5
After 5 iterations the residual is 2.3504720185146307e-6
After 6 iterations the residual is 8.531230359867334e-8
After 7 iterations the residual is 3.116955234295595e-9
After 8 iterations the residual is 1.194485773402653e-10
After 9 iterations the residual is 4.563823474262472e-12
After 10 iterations the residual is 1.9658699853215524e-13
  2.450691 seconds (60.12 k allocations: 8.539 GiB, 23.35% gc time)


In [51]:
#base_length = [16,16,16]
@time guess, residuals = multigrid_V_cycle(base_length; num_iters=[10, 1, 1, 1, 1]);

After 1 iterations the residual is 1.0
After 2 iterations the residual is 0.059735493832536034
After 3 iterations the residual is 0.002053590831796095
After 4 iterations the residual is 7.163604481865302e-5
After 5 iterations the residual is 2.562772744520645e-6
After 6 iterations the residual is 9.520613201570942e-8
After 7 iterations the residual is 3.5353486979468325e-9
After 8 iterations the residual is 1.4443515771764684e-10
After 9 iterations the residual is 6.0048602724627685e-12
After 10 iterations the residual is 2.9945256392252015e-13
 95.837082 seconds (72.13 k allocations: 368.214 GiB, 3.84% gc time)


In [101]:
u = np.zeros([65, 65, 65])
set_boundary_conditions(u)
rhs = -apply_stencil(u)
guess = np.zeros_like(u)


  0.002532 seconds


Array{Float64, 3}

In [19]:
u=np.zeros([5,5,5])
set_boundary_conditions(u)
rhs = -apply_stencil(u)
rhs_flat = reshape(rhs[2:end-1, 2:end-1, 2:end-1], :)

linear_function = x_flat -> matvec_shape(x_flat, (3,3,3); factor = 1)

A_op = FunctionMap{Float64,false}(linear_function, 3*3*3)

x = cg(A_op, rhs_flat)

y=reshape(x, (3,3,3) )
#x = linear_function(rhs_flat)
#guess = np.zeros_like(u)

#rb_gauss_seidel_3d(guess, rhs, diag=1, iterations=3)

#new = restrict(guess)
#refined = prolong(new)

#vec(new)

u[2:end-1, 2:end-1, 2:end-1]=y
u

5×5×5 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25
 0.25  0.25  0.25  0.25  0.25

[:, :, 3] =
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5
 0.5  0.5  0.5  0.5  0.5

[:, :, 4] =
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75
 0.75  0.75  0.75  0.75  0.75

[:, :, 5] =
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0

In [1]:
Threads.nthreads()

10